# SORT1 enhancer — 630 bp GFP/reporter region swap (K562)

Reproduces the MCP `analyze_region_swap` call. Replaces the
genomic region **chr1:109274500-109275500** with a custom DNA sequence
and scores the same regulatory layers the variant API uses.


In [ ]:
# All imports the notebook needs — top-level, no later imports.
import json
from pathlib import Path

import chorus
from chorus.analysis.normalization import get_normalizer
from chorus.analysis.region_swap import analyze_region_swap
from chorus.analysis.analysis_request import AnalysisRequest

In [ ]:
WALKTHROUGH_DIR = Path.cwd()
print(f"Writing artifacts to: {WALKTHROUGH_DIR}")

In [ ]:
oracle = chorus.create_oracle(
    oracle_name='alphagenome',
    use_environment=True,
)
oracle.load_pretrained_model()

In [ ]:
# Inputs.
oracle_name = 'alphagenome'
region = 'chr1:109274500-109275500'
gene_name = 'SORT1'
assay_ids = [
    'DNASE/EFO:0002067 DNase-seq/.',
    'CHIP_HISTONE/EFO:0002067 Histone ChIP-seq H3K27ac/.',
    'CHIP_HISTONE/EFO:0002067 Histone ChIP-seq H3K4me3/.',
    'CAGE/hCAGE EFO:0002067/+',
]
replacement_sequence = (
        "GCCACCATGGCCACCATGGCCACCATGGCCACCATGGCCACCATGGCCACCATGGCCACCATGC"
        "GAATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCCTGGTCGAGCTGGACGG"
        "CGACGTAAACGGCCACAAGTTCAGCGTGTCCGGCGAGGGCGAGGGCGATGCCACCTACGGCAAG"
        "CTGACCCTGAAGTTCATCTGCACCACCGGCAAGCTGCCCGTGCCCTGGCCCACCCTCGTGACCA"
        "CCCTGACCTACGGCGTGCAGTGCTTCAGCCGCTACCCCGACCACATGAAGCAGCACGACTTCTT"
        "CAAGTCCGCCATGCCCGAAGGCTACGTCCAGGAGCGCACCATCTTCTTCAAGGACGACGGCAAC"
        "TACAAGACCCGCGCCGAGGTGAAGTTCGAGGGCGACACCCTGGTGAACCGCATCGAGCTGAAGG"
        "GCATCGACTTCAAGGAGGACGGCAACATCCTGGGGCACAAGCTGGAGTACAACTACAACAGCCA"
        "CAACGTCTATATCATGGCCGACAAGCAGAAGAACGGCATCAAGGTGAACTTCAAGATCCGCCAC"
        "AACATCGAGGACGGCAGCGTGCAGCTCGCCGACCACTACCAGCAGAACACCCCC"
    )
print(f"Replacement length: {len(replacement_sequence)} bp")

In [ ]:
# Score the swap. Returns a VariantReport with ref (WT) vs alt
# (post-replacement) predictions in each layer.
normalizer = get_normalizer(oracle_name=oracle_name)
analysis_request = AnalysisRequest(
    user_prompt=(
        f"Replace {region} with a {len(replacement_sequence)} bp "
        f"construct and predict effects on K562 tracks."
    ),
    tool_name="analyze_region_swap",
    oracle_name=oracle_name,
    tracks_requested=f"{len(assay_ids)} K562 tracks",
)
report = analyze_region_swap(
    oracle=oracle,
    region=region,
    replacement_sequence=replacement_sequence,
    assay_ids=assay_ids,
    gene_name=gene_name,
    normalizer=normalizer,
    oracle_name=oracle_name,
)
report.analysis_request = analysis_request

In [ ]:
# Save the same artifacts the MCP tool would produce:
#   - example_output.md  (markdown report)
#   - example_output.json (structured scores)
#   - example_output.tsv (track-level table)
#   - region_swap_SORT1_K562_report.html (interactive IGV report)
WALKTHROUGH_DIR.joinpath("example_output.md").write_text(report.to_markdown())
WALKTHROUGH_DIR.joinpath("example_output.json").write_text(
    json.dumps(report.to_dict(), indent=2, default=str)
)
try:
    report.to_dataframe().to_csv(
        WALKTHROUGH_DIR / "example_output.tsv", sep="\t", index=False,
    )
except Exception as exc:
    print(f"TSV write skipped: {exc}")

_html_path = report.to_html(output_path=str(WALKTHROUGH_DIR / "region_swap_SORT1_K562_report.html"))
print(f"Wrote artifacts to { WALKTHROUGH_DIR }")


## What this notebook produced

- `example_output.md/.json/.tsv` — WT vs replacement per-layer scores
- `region_swap_SORT1_K562_report.html` — IGV browser with WT/replacement track overlay
